# Train YOLO11L Artifact Detector 

This notebook trains YOLO11L to detect `pyramid`, `tutankhamun_mask`, and `nefertiti_head` using Kaggle inputs.

In [8]:
import sys
print("Python:", sys.executable)

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

Python: /usr/bin/python3
Torch: 2.10.0+cu128
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


## 2. Install / Check Ultralytics

In [9]:
try:
    import ultralytics
    print("ultralytics:", ultralytics.__version__)
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])
    import ultralytics
    print("ultralytics:", ultralytics.__version__)

ultralytics: 8.4.50


## 3. Set Kaggle Paths

These paths match the inputs shown in your Kaggle sidebar.

In [10]:
from pathlib import Path
from ultralytics import YOLO

DATASET_INPUT = Path("/kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset")
MODEL_INPUT = Path("/kaggle/input/models/abdelmonemhatem/yollo11l/pytorch/default/1")

DATASET_DIR = DATASET_INPUT / "dataset"
DATA_PATH = DATASET_DIR / "data_train.yaml"
MODEL_PATH = MODEL_INPUT / "yolo11l.pt"
RUNS_DIR = Path("/kaggle/working/runs")

print("Dataset input exists:", DATASET_INPUT.exists(), DATASET_INPUT)
print("Dataset dir exists:", DATASET_DIR.exists(), DATASET_DIR)
print("Data YAML exists:", DATA_PATH.exists(), DATA_PATH)
print("Model input exists:", MODEL_INPUT.exists(), MODEL_INPUT)
print("Model exists:", MODEL_PATH.exists(), MODEL_PATH)
print("Runs dir:", RUNS_DIR)

assert DATASET_INPUT.exists(), f"Missing dataset input: {DATASET_INPUT}"
assert DATASET_DIR.exists(), f"Missing dataset folder: {DATASET_DIR}"
assert DATA_PATH.exists(), f"Missing data yaml: {DATA_PATH}"
assert MODEL_INPUT.exists(), f"Missing model input: {MODEL_INPUT}"
assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}"


Dataset input exists: True /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset
Dataset dir exists: True /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset
Data YAML exists: True /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset/data_train.yaml
Model input exists: True /kaggle/input/models/abdelmonemhatem/yollo11l/pytorch/default/1
Model exists: True /kaggle/input/models/abdelmonemhatem/yollo11l/pytorch/default/1/yolo11l.pt
Runs dir: /kaggle/working/runs


## 4. Write Kaggle Training YAML

The uploaded `data_train.yaml` may contain a Windows path. This cell creates a Kaggle-specific YAML with the correct `/kaggle/input` path.

In [11]:
KAGGLE_DATA_YAML = Path("/kaggle/working/data_kaggle.yaml")
KAGGLE_DATA_YAML.write_text(
    "\n".join([
        f"path: {DATASET_DIR}",
        "train: images/train",
        "val: images/val",
        "test: images/test",
        "",
        "names:",
        "  0: pyramid",
        "  1: tutankhamun_mask",
        "  2: nefertiti_head",
        "",
    ]),
    encoding="utf-8",
)
print(KAGGLE_DATA_YAML.read_text())

path: /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset
train: images/train
val: images/val
test: images/test

names:
  0: pyramid
  1: tutankhamun_mask
  2: nefertiti_head



## 5. Check Dataset Files

In [12]:
for split in ["train", "val", "test"]:
    images = list((DATASET_DIR / "images" / split).glob("*"))
    labels = list((DATASET_DIR / "labels" / split).glob("*.txt"))
    print(split, "images:", len(images), "labels:", len(labels))

train images: 506 labels: 506
val images: 144 labels: 144
test images: 74 labels: 74


## 6. Load YOLO11L

In [13]:
model = YOLO(str(MODEL_PATH))
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_

## 7. Train

Start with `batch=8` on Kaggle T4. If it runs out of memory, change to `batch=4`.

In [14]:
results = model.train(
    data=str(KAGGLE_DATA_YAML),
    epochs=80,
    imgsz=640,
    batch=8,
    project=str(RUNS_DIR),
    name="artifact_yolo11l",
    task="detect",
    device=0,
    workers=2,
)

Ultralytics 8.4.50 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_kaggle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/models/abdelmonemhatem/yollo11l/pytorch/default/1/yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=artifact_yolo11l, nbs=64, nms=False, op

## 8. Validate Best Model

In [15]:
best_model_path = RUNS_DIR / "artifact_yolo11l" / "weights" / "best.pt"
print("Best model exists:", best_model_path.exists(), best_model_path)

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(KAGGLE_DATA_YAML), split="val", device=0)
metrics

Best model exists: True /kaggle/working/runs/artifact_yolo11l/weights/best.pt
Ultralytics 8.4.50 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11l summary (fused): 191 layers, 25,281,625 parameters, 0 gradients, 86.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 361.5±96.2 MB/s, size: 184.4 KB)
val: Scanning /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset/labels/val... 144 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 144/144 1.0Kit/s 0.1s<0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.4it/s 3.8s0.4ss
                   all        144        166      0.995      0.993      0.995      0.956
               pyramid         44         44      0.997          1      0.995      0.989
      tutankhamun

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bcf60164230>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

## 9. Predict on Test Images

In [16]:
predictions = best_model.predict(
    source=str(DATASET_DIR / "images" / "test"),
    conf=0.5,
    save=True,
    project=str(RUNS_DIR),
    name="test_predictions",
    device=0,
)
print("Prediction output:", RUNS_DIR / "test_predictions")


image 1/74 /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset/images/test/mixed_mixed_multi_object_mixed_multi_object_20260514_020507_270_jpg.rf.H16XsjC1nk5ARmv8CGAb.jpg: 384x640 1 pyramid, 1 nefertiti_head, 59.5ms
image 2/74 /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset/images/test/mixed_mixed_multi_object_mixed_multi_object_20260514_023738_767_jpg.rf.1e1Dv8S06fkpMNsQvk2O.jpg: 384x640 1 pyramid, 1 tutankhamun_mask, 1 nefertiti_head, 29.0ms
image 3/74 /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset/images/test/mixed_mixed_multi_object_mixed_multi_object_20260514_023739_515_jpg.rf.VSvGkD0DeIe4uRhHrDA4.jpg: 384x640 1 pyramid, 1 tutankhamun_mask, 1 nefertiti_head, 26.2ms
image 4/74 /kaggle/input/datasets/abdelmonemhatem/egyptian-artifacts-yolo11-dataset/dataset/images/test/mixed_mixed_multi_object_mixed_multi_object_20260514_023740_642_jpg.rf.wwdy7NZgNZwhMOZtolmW.jpg: 384x640 1 pyramid, 1 tu

## 10. Copy Best Model to Working Folder

This makes the model easy to download from Kaggle output.

In [17]:
import shutil

final_model = Path("/kaggle/working/artifact_yolo11l_best.pt")
shutil.copy2(best_model_path, final_model)
print("Download this model:", final_model)

Download this model: /kaggle/working/artifact_yolo11l_best.pt
